In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif  # Import SelectKBest and f_classif
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency
import seaborn as sns
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Task 1: Write your code here:

df_quiz = pd.read_csv(f"{path}/Q1_data.csv")
print(f"Dataset shape: {df_quiz.shape}")

In [ ]:
# Task 2: Write your code here:
df_quiz.head()

In [ ]:
# Task 3: Write your code here:
df_quiz.info()

In [ ]:
# Task 4: Write your code here:
df_quiz.describe()


In [ ]:
# Task 5: Write your code here:
# Target distribution
condition_counts = df_quiz['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(condition_counts.index, condition_counts.values, color='coral')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task 1: Write your code here:
#df_quiz = df_quiz.columns.drop("Order_ID")
df_quiz= df_quiz.drop('Order_ID', axis=1)

In [ ]:
print(f"Dataset shape: {df_quiz.shape}")

In [ ]:
# Task 2: Write your code here:
print("\nMissing Values (df.isnull().sum()):")
print(df_quiz.isnull().sum())

In [ ]:
mode_values = df_quiz.mode().iloc[0]
print(mode_values)

In [ ]:
# Analyze missing values
missing_percentage = (df_quiz.isnull().sum() / len(df_quiz)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
cols = ['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']
df_quiz = df_quiz[cols].copy()

df_quiz= df_quiz.dropna(subset=['Delivery_Time']) #bc rows with no target labels are useless
df_quiz['Weather'].fillna(df_quiz['Weather'].mode()[0]) #we take the avg redundunt weather for the reagion
df_quiz['Traffic_Level'].fillna(df_quiz['Traffic_Level'].mode()[0]) #we take the avg redundunt Trafic level for the reagion
df_quiz['Time_of_Day'].fillna(df_quiz['Time_of_Day'].mode()[0]) #we take the avg redundunt time of day
df_quiz['Courier_Experience_yrs'].fillna(0) #bc people usually leave fields empty if they dont have entry (experience) do i assumed they dont have experience


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df_quiz):
  duplicates = df_quiz.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_quiz.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_quiz)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df_quiz.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

# Encode the target column
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_quiz[col] = le.fit_transform(df_quiz[col])
  label_encoders[col] = le

df_quiz

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_quiz.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_quiz[numerical_cols] = scaler.fit_transform(df_quiz[numerical_cols])
df_quiz.head()


In [ ]:
# Task 6: Write your code here:
#since it is multi regressign task and delevery time can ranges then it does not need to be distributed

In [ ]:
def check_target_imbalance(df_quiz, target_column):
  print("Target Distribution:")
  print(df_quiz[target_column].value_counts(normalize=True))
  sns.countplot(x=df_quiz[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_quiz, "Delivery_Time")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Import models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')



In [ ]:
# Task 1: Write your code here:
# Define features (X) and target (y)
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df_quiz[feature_cols]
y = df_quiz['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
sklearn_models = {
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
  #randomforest model
    model= RandomForestClassifier()

    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
df_quiz['Delivery_Time'].plot.hist(bins=30, color='blue', title='Delivery_Time')

In [ ]:
# Task Bonus: Write your code here:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = sklearn_mae(y_test, y_pred)
    # Store results
    all_results[model_name]["mae"].append(mse)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mse']):.4f}")